# Movie Recommendation System - Exploratory Data Analysis (EDA)

This notebook demonstrates the step-by-step Exploratory Data Analysis (EDA) and prototype testing of the **Movie Recommendation System** models.

In [ ]:
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Add root directory to sys.path to access project modules
sys.path.append(os.path.abspath('..'))
import config
from utils.preprocessing import load_and_preprocess_all
from models.content_filter import ContentRecommender
from models.collaborative_filter import CollaborativeRecommender
from models.hybrid_model import HybridRecommender

print("Libraries loaded successfully.")

## 1. Load and Inspect the Data

We load the movies (with combined tags), ratings, and tags dataframes using our preprocessing utility.

In [ ]:
movies_df, ratings_df, tags_df = load_and_preprocess_all()
print(f"Movies shape: {movies_df.shape}")
print(f"Ratings shape: {ratings_df.shape}")
print(f"Tags shape: {tags_df.shape}")

Let's look at the first few rows of the movie dataframe:

In [ ]:
movies_df.head()

## 2. Exploratory Visualizations

### Genre Distribution
Let's count how many movies exist in each genre.

In [ ]:
genres_exploded = movies_df['genres'].str.split('|').explode()
plt.figure(figsize=(12, 6))
sns.countplot(y=genres_exploded, order=genres_exploded.value_counts().index, palette='viridis')
plt.title('Movie Count by Genre')
plt.xlabel('Count')
plt.ylabel('Genre')
plt.show()

### Rating Frequency Distribution

In [ ]:
plt.figure(figsize=(8, 5))
sns.histplot(ratings_df['rating'], bins=10, kde=True, color='crimson')
plt.title('Distribution of Movie Ratings')
plt.xlabel('Rating value')
plt.ylabel('Frequency')
plt.show()

## 3. Test Content-Based Recommender

In [ ]:
content_rec = ContentRecommender()
content_rec.fit(movies_df)

# Search similar movies to 'Toy Story (1995)' - Movie ID 1
toy_story_id = 1
recs = content_rec.recommend_similar_movies(toy_story_id, top_n=5)

print(f"Recommendations similar to '{movies_df[movies_df['movieId'] == toy_story_id]['title'].values[0]}':")
for r_id, score in recs:
    title = movies_df[movies_df['movieId'] == r_id]['title'].values[0]
    print(f"- {title} (Similarity Score: {score:.3f})")

## 4. Test Collaborative Filtering Recommender

In [ ]:
collab_rec = CollaborativeRecommender(kind="item")
collab_rec.fit(ratings_df)

# Mock User ratings profile: likes Toy Story (movieId 1) and Matrix (movieId 2571)
mock_profile = {1: 5.0, 2571: 5.0}
recs_collab = collab_rec.recommend_for_user(mock_profile, top_n=5)

print("Item-Based Collaborative Filtering Recommendations:")
for r_id, score in recs_collab:
    title = movies_df[movies_df['movieId'] == r_id]['title'].values[0]
    print(f"- {title} (Predicted Rating: {score:.2f})")

## 5. Test Hybrid Recommender

In [ ]:
hybrid_rec = HybridRecommender(collab_kind="item")
hybrid_rec.fit(movies_df, ratings_df)

recs_hybrid = hybrid_rec.recommend_hybrid(mock_profile, w_content=0.5, w_collab=0.5, top_n=5)

print("Hybrid Model Recommendations:")
for rec in recs_hybrid:
    print(f"- {rec['title']} (Hybrid Score: {rec['hybrid_score']*100:.1f}%, Reason: {rec['reason']})")